## Dependencies

In [ ]:
from openpyxl import load_workbook
import pandas as pd
import re
from pyspark.sql import functions as F

## Parametros

In [ ]:

source_path = "/Volumes/main/landing/clientes.xlsx"
sheet_name = "Clientes"

target_table = "catalog.dimensions.dim_clientes"

primary_key = "id_cliente"

# Solo se utilizan en SCD2
version_columns = ["ciudad", "categoria"]

load_strategy = "SCD2"  # SCD1 / SCD2 / REPLACE

process_operator = "Databricks_Job"

## File Read 
#### Esta sección utiliza openpyxl en modo read_only para evitar cargar innecesariamente el libro completo en memoria. También contempla filas vacías y encabezados.

In [ ]:
workbook = load_workbook(
    source_path,
    read_only=True,
    data_only=True
)

worksheet = workbook[sheet_name]

rows = worksheet.iter_rows(values_only=True)

headers = next(rows)

# Eliminar espacios de los encabezados
headers = [str(col).strip() if col is not None else "" for col in headers]

data = [
    row for row in rows
    if any(value is not None for value in row)
]

workbook.close()

pdf = pd.DataFrame(data, columns=headers)

if pdf.empty:
    raise ValueError("El archivo Excel no contiene registros.")

print(f"Registros leídos: {len(pdf)}")
print(f"Columnas detectadas: {len(pdf.columns)}")

## Normalización e inferencia de tipos
#### Normalizamos los nombres y dejamos que Spark infiera los tipos a partir de los valores leídos. Para columnas completamente vacías, asignamos temporalmente valores vacíos para que Spark pueda inferirlas como texto y luego los convertimos a NULL.

In [ ]:

import re
from pyspark.sql import functions as F

def normalize_column(name):
    name = name.strip().lower()
    name = re.sub(r"[^a-zA-Z0-9]+", "_", name)
    return name.strip("_")


pdf.columns = [normalize_column(col) for col in pdf.columns]

if len(pdf.columns) != len(set(pdf.columns)):
    raise ValueError("Hay nombres de columnas duplicados después de normalizar.")

# Columnas completamente vacías
empty_columns = [
    col for col in pdf.columns
    if pdf[col].isna().all()
]

for col in empty_columns:
    pdf[col] = ""

df = spark.createDataFrame(pdf)

# Restaurar como NULL las columnas que estaban vacías
for col in empty_columns:
    df = df.withColumn(
        col,
        F.lit(None).cast("string")
    )

# Validaciones básicas
required_columns = [primary_key] + (
    version_columns if load_strategy == "SCD2" else []
)

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

if df.filter(F.col(primary_key).isNull()).limit(1).count() > 0:
    raise ValueError("La clave primaria contiene valores NULL.")

if df.groupBy(primary_key).count().filter("count > 1").limit(1).count() > 0:
    raise ValueError("El Excel contiene claves primarias duplicadas.")

print("Esquema detectado:")
df.printSchema()